In [71]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sb
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, precision_recall_curve
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler

In [73]:
def ModelPreProcessing(df):

    # Outlier treatment
    df_proc = df[(df['person_age'] <= 90) & (df['person_emp_length'] <= 45)].copy()
    
    # Missing values treatment
    df_proc.fillna({'loan_int_rate': df_proc['loan_int_rate'].median()}, inplace=True)
    df_proc = df_proc.drop(columns=['loan_grade'], errors='ignore')

    # Categorical values treatment
    person_home_ownership = pd.get_dummies(df_proc['person_home_ownership'], drop_first=True).astype(int)
    loan_intent = pd.get_dummies(df_proc['loan_intent'], drop_first=True).astype(int)
    
    # Scaling
    data_to_scale = df_proc.drop(columns=['person_home_ownership', 'loan_intent', 'cb_person_default_on_file', 'loan_status'], errors='ignore')
    scaler = StandardScaler()
    scaled_array = scaler.fit_transform(data_to_scale)
    scaled_numeric_df = pd.DataFrame(scaled_array, columns=data_to_scale.columns, index=df_proc.index)
    
    # featues & target
    features = pd.concat([scaled_numeric_df, person_home_ownership, loan_intent], axis=1)
    features['cb_person_default_on_file'] = np.where(df_proc['cb_person_default_on_file'] == 'Y', 1, 0)
    
    # SMOTE - Imbalance treatement
    smote = SMOTE(random_state=42)
    balanced_features, balanced_target = smote.fit_resample(features, target)
    
    return target, features, balanced_features, balanced_target, data_to_scale
